# Shogi AI Lab — Colab 学習ノートブック（GitHub clone 版）

AlphaZero 方式（policy/value ネット + MCTS）で将棋AIを学習します。
このノートは Drive にリポジトリを置く代わりに、**GitHub の作業ブランチを直接 clone** して使います。clone するブランチは
実行時点で最新の `mochi/batch-*` を自動で選ぶので、第3弾以降の改良が
push されていれば、その最新版で学習が回ります（選ばれたブランチと
コミットはセルの出力に表示されます）。

第3弾・第4弾の機能はこのノートの設定で有効化しています：
自己対局は **Gumbel 探索＋playout cap** で回し、cshogi の特徴量生成は
ネイティブ経路（文字列経由なし）になっています。
Google Drive は成果物（チェックポイントと `data/learn/`）の保存先としてだけ使います。

**このノートブックの流れ**

1. GPU を確認する
2. `cshogi` と PyTorch を入れる
3. リポジトリを GitHub から clone する（Drive は成果物用にマウント）
4. **テストを走らせて cshogi バックエンドの正しさを検証する**（重要）
5. 速度を実測して、1ラウンドの局数を決める
6. 既存データで事前学習する（任意・推奨）
7. 自己対局 → 学習 → ゲーティング対戦のループを回す
8. 強さを測って、成果物を Drive に保存する

> **速度の前提**: 自己対局のボトルネックは GPU ではなく CPU です。
> 合法手生成と MCTS のツリー操作が律速するため、`--processes` を
> vCPU 数に合わせることが GPU を大きくするより効きます。


## 1. GPU を確認する

In [1]:
!nvidia-smi
import torch, os
print('torch    :', torch.__version__)
print('cuda     :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu      :', torch.cuda.get_device_name(0))
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'vram     : {total:.1f} GB')
print('vcpu     :', os.cpu_count())

Fri Nov 13 01:53:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   44C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. 依存を入れる

`cshogi` は C++ 実装の将棋ライブラリです。合法手生成が純Python比で3桁速く、
自己対局を現実的な時間に収めるために必須です。

sdist からビルドされるので、**出力は省略しません**。失敗したらここで分かります。

> `import cshogi` が失敗する原因は「未インストール」だけではありません。
> Colab でよくあるのは NumPy の ABI 不一致です。その場合は
> **ランタイムを再起動してからこのセルをもう一度**実行してください。

In [2]:
# 出力を省略しない。ビルドエラーはここに出る
!pip install --no-cache-dir cshogi


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 182.5 MB/s  0:00:00


In [3]:
# インストールできたかを厳しく確認する。ここを通らなければ先に進まない
import importlib, subprocess, sys

import numpy
print('python :', sys.version.split()[0])
print('numpy  :', numpy.__version__)

try:
    cshogi = importlib.import_module('cshogi')
except ImportError as exc:
    print('\n*** cshogi をインポートできません ***')
    print('理由:', exc)
    print()
    if 'numpy' in str(exc).lower():
        print('NumPy の ABI 不一致です。ランタイムを再起動してから')
        print('このノートブックをセル2から再実行してください。')
    else:
        print('上のインストール出力にビルドエラーが出ていないか確認してください。')
    print()
    print('pip の認識:')
    print(subprocess.run([sys.executable, '-m', 'pip', 'show', 'cshogi'],
                         capture_output=True, text=True).stdout or '  (未インストール)')
    raise SystemExit('cshogi が使えないため中断しました')

print('cshogi :', getattr(cshogi, '__version__', '(版数不明)'))
print('board  :', cshogi.Board().sfen())

# サブプロセスからも見えることを確認する。learn_loop は各段階を
# サブプロセスで実行するため、ここが通らないと学習ループで失敗する
check = subprocess.run([sys.executable, '-c', 'import cshogi; print(cshogi.Board().sfen())'],
                       capture_output=True, text=True)
assert check.returncode == 0, f'サブプロセスで失敗: {check.stderr}'
print('subprocess から見えることも確認:', check.stdout.strip())


python : 3.13.16
numpy  : 2.4.6
cshogi : (版数不明)
board  : lnsgkgsnl/1r5b1/ppppppppp/9/9/9/PPPPPPPPP/1B5R1/LNSGKGSNL b - 1
subprocess から見えることも確認: lnsgkgsnl/1r5b1/ppppppppp/9/9/9/PPPPPPPPP/1B5R1/LNSGKGSNL b - 1


## 3. リポジトリを読み込む

GitHub から作業ブランチを clone します。あわせて、成果物の保存先として
Google Drive をマウントします（学習結果はランタイムが消えても残るよう、
最後に Drive へコピーします）。


In [4]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [5]:
import os, subprocess

# 学習に使うコードは、その時点で GitHub にある最新の mochi/batch-* ブランチ。
# 第3弾以降が push されれば、ノートブックを直さなくても自動で切り替わる。
_out = subprocess.run(
    ['git', 'ls-remote', '--heads', 'https://github.com/hirotakaster/shogi-ai-lab'],
    capture_output=True, text=True).stdout
_branches = [line.split('refs/heads/')[-1].strip() for line in _out.splitlines()
             if 'refs/heads/mochi/batch-' in line]

def _batch_no(name):
    try:
        return int(name.split('batch-')[1].split('-')[0])
    except Exception:
        return -1

BRANCH = max(_branches, key=_batch_no) if _branches else 'mochi/batch-2-search-data'
print('training code branch:', BRANCH)

ARTIFACTS = '/content/drive/MyDrive/shogi-ai-lab-artifacts'
os.makedirs(ARTIFACTS, exist_ok=True)

if not os.path.isdir('/content/shogi-ai-lab'):
    !git clone --branch {BRANCH} --single-branch https://github.com/hirotakaster/shogi-ai-lab /content/shogi-ai-lab
else:
    print('already cloned; 最新化する場合はこのフォルダを消して再実行してください')

!cd /content/shogi-ai-lab && git log --oneline -1


training code branch: mochi/batch-5-scale-stability
Cloning into '/content/shogi-ai-lab'...
remote: Enumerating objects: 134, done.
remote: Counting objects: 100% (92/92), done.
remote: Compressing objects: 100% (87/87), done.
remote: Total 134 (delta 44), reused 14 (delta 5), pack-reused 42 (from 1)
Receiving objects: 100% (134/134), 66.04 MiB | 18.83 MiB/s, done.
Resolving deltas: 100% (44/44), done.
85fee2c (HEAD -> mochi/batch-5-scale-stability, origin/mochi/batch-5-scale-stability) Batch 5 (Mochi): finetuning regime (--round-lr/--round-epochs), newest-round oversampling (--latest-repeat), --grad-clip forwarding, gating in self-play search mode; production Colab recipe (L4) in AGENTS.md; tests 129 -> 137


In [6]:
import os, sys

REPO = '/content/shogi-ai-lab'
assert os.path.isdir(REPO), f'not found: {REPO}'
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
print('cwd:', os.getcwd())
print(sorted(os.listdir('shogi_ai')))

# ランタイムを再起動すると cwd は /content に戻り、REPO も未定義になります。
# 以降のセルは `!cd {REPO} && python -m ...` の形で REPO を参照するので、
# 再起動したらこのセルを必ず先に再実行してください。


cwd: /content/shogi-ai-lab
['__init__.py', 'backends.py', 'core.py', 'encoding.py', 'engine.py', 'evaluate.py', 'evaluator.py', 'learn_loop.py', 'mcts.py', 'network.py', 'nn_model.py', 'self_play.py', 'selfplay_az.py', 'train_az.py']


## 4. テストで cshogi バックエンドを検証する

`tests/test_cshogi_backend.py` は、cshogi バックエンドが純Python実装と
**同じ特徴量・同じ合法手・同じ方策インデックス**を出すことを確認します。
cshogi は Apple Silicon でビルドできないため、この検証は Colab でしか走りません。

ここが失敗したら、学習を始める前に必ず直してください。
スキップされた場合は cshogi が入っていません。

In [7]:
!cd {REPO} && python -m unittest discover -s tests -v 2>&1 | tail -30

test_gumbel_games_produce_valid_records (test_selfplay_cap.GumbelSelfPlayTests.test_gumbel_games_produce_valid_records) ... ok
test_capped_budget_defaults_to_a_quarter (test_selfplay_cap.PlayoutCapTests.test_capped_budget_defaults_to_a_quarter) ... ok
test_capped_plies_are_played_but_not_recorded (test_selfplay_cap.PlayoutCapTests.test_capped_plies_are_played_but_not_recorded) ... ok
test_disabled_cap_records_every_ply (test_selfplay_cap.PlayoutCapTests.test_disabled_cap_records_every_ply) ... ok
test_full_fraction_records_every_ply (test_selfplay_cap.PlayoutCapTests.test_full_fraction_records_every_ply) ... ok
test_invalid_fraction_is_rejected (test_selfplay_cap.PlayoutCapTests.test_invalid_fraction_is_rejected) ... ok
test_capped_record_uses_the_search_value (test_training_data.CappedRecordTests.test_capped_record_uses_the_search_value) ... ok
test_capped_record_without_a_search_value_falls_back (test_training_data.CappedRecordTests.test_capped_record_without_a_search_value_falls_bac

In [8]:
# cshogi 側だけを個別に確認する（skip が出ていないことを見る）
!cd {REPO} && python -m unittest tests.test_cshogi_backend -v 2>&1 | tail -20

test_pop_restores_the_position (tests.test_cshogi_backend.CshogiAgreementTests.test_pop_restores_the_position) ... ok
test_search_runs_on_the_fast_backend (tests.test_cshogi_backend.CshogiAgreementTests.test_search_runs_on_the_fast_backend) ... ok
test_self_play_produces_records (tests.test_cshogi_backend.CshogiAgreementTests.test_self_play_produces_records) ... ok
test_sfen_agrees_along_a_line (tests.test_cshogi_backend.CshogiAgreementTests.test_sfen_agrees_along_a_line) ... ok
test_snapshots_and_planes_agree (tests.test_cshogi_backend.CshogiAgreementTests.test_snapshots_and_planes_agree) ... ok
test_terminal_value_is_none_while_the_game_is_live (tests.test_cshogi_backend.CshogiAgreementTests.test_terminal_value_is_none_while_the_game_is_live) ... ok
test_turn_mapping (tests.test_cshogi_backend.CshogiAgreementTests.test_turn_mapping) ... ok
test_capture_line_fills_both_hands (tests.test_cshogi_backend.FixtureTests.test_capture_line_fills_both_hands)
Otherwise the hand planes would nev

### 4b. cshogi の API 形状を検査する

cshogi のバージョン差で関数の置き場所が変わることがあります（`move_to_usi` は
モジュール関数で、`Board` のメソッドではありません）。

このセルは依存している呼び出しを**全部まとめて**検査し、合わない点を一覧で出します。
1つずつ実行時エラーで潰すより速く、特に**手番の色が反転していないか**を
純Python実装と照合します。ここが狂うと学習データの符号が全部反転します。

In [9]:
!cd {REPO} && python -m shogi_ai.backends


cshogi backend OK (cshogi (version unknown))


## 5. 速度を実測する

1局あたりの所要時間から、1ラウンドに何局回せるかを決めます。

In [10]:
import time
import numpy as np
from shogi_ai.backends import CshogiBackend, PurePythonBackend
from shogi_ai.evaluator import TorchEvaluator, UniformEvaluator
from shogi_ai.mcts import MCTS
from shogi_ai.network import build_network, describe

# 合法手生成の速度差
for name, backend in (('python', PurePythonBackend()), ('cshogi', CshogiBackend())):
    start, calls = time.time(), 0
    while time.time() - start < 1.0:
        backend.legal_moves(); calls += 1
    print(f'{name:7s} legal_moves: {calls:7d}/s')

# MCTS の実効速度
PRESET = 'base'
model = build_network(PRESET)
print(PRESET, describe(model))
evaluator = TorchEvaluator(model, device='cuda')
for batch in (8, 16, 32, 64):
    backend = CshogiBackend()
    mcts = MCTS(evaluator, simulations=400, batch_size=batch)
    start = time.time(); mcts.run(backend); elapsed = time.time() - start
    print(f'batch {batch:3d}: {400/elapsed:7.0f} sims/s')

python  legal_moves:     405/s
cshogi  legal_moves:   57974/s
base 10 blocks x 192ch, 7.47M params
batch   8:     410 sims/s
batch  16:    1853 sims/s
batch  32:    2444 sims/s
batch  64:    3158 sims/s


In [11]:
# 1局まるごと計測して、必要な時間を見積もる
from shogi_ai.selfplay_az import SelfPlayConfig, play_one_game

SIMULATIONS = 200
MCTS_BATCH = 32
config = SelfPlayConfig(simulations=SIMULATIONS, batch_size=MCTS_BATCH,
                        backend='cshogi', max_moves=320,
                        search_mode='gumbel', playout_cap_fraction=0.25)
start = time.time()
summary = play_one_game(evaluator, config, seed=0)
elapsed = time.time() - start
print(f'1 game: {elapsed:.1f}s, {summary.plies} plies, '
      f'{len(summary.records)} positions, reason={summary.reason}')
workers = os.cpu_count()
print(f'\n{workers} 並列なら 1時間あたり約 {3600 / elapsed * workers:.0f} 局')

1 game: 28.4s, 178 plies, 52 positions, reason=mate

12 並列なら 1時間あたり約 1522 局


## 6. 既存データで事前学習する（推奨）

従来型エンジン同士の対局で初期データを作り、教師あり学習でネットを
温めます。ランダム初期化から始めるより、最初の数ラウンドが目に見えて
速く進みます。すでに `data/initial_selfplay.jsonl` があれば生成は飛ばします。


In [12]:
import os
if os.path.exists('data/initial_selfplay.jsonl'):
    print('initial data already exists')
else:
    !cd {REPO} && python -m shogi_ai.self_play --games 40 --depth 2 --processes {os.cpu_count()} --out data/initial_selfplay.jsonl


game 1/40: 27 positions
game 2/40: 30 positions
game 3/40: 32 positions
game 4/40: 44 positions
game 5/40: 37 positions
game 6/40: 71 positions
game 7/40: 106 positions
game 8/40: 114 positions
game 9/40: 122 positions
game 10/40: 137 positions
game 11/40: 41 positions
game 12/40: 108 positions
game 13/40: 180 positions
game 14/40: 69 positions
game 15/40: 180 positions
game 16/40: 31 positions
game 17/40: 180 positions
game 18/40: 155 positions
game 19/40: 180 positions
game 20/40: 120 positions
game 21/40: 163 positions
game 22/40: 180 positions
game 23/40: 51 positions
game 24/40: 180 positions
game 25/40: 161 positions
game 26/40: 33 positions
game 27/40: 166 positions
game 28/40: 180 positions
game 29/40: 99 positions
game 30/40: 148 positions
game 31/40: 152 positions
game 32/40: 180 positions
game 33/40: 109 positions
game 34/40: 180 positions
game 35/40: 180 positions
game 36/40: 180 positions
game 37/40: 180 positions
game 38/40: 63 positions
game 39/40: 180 positions
game 40/

In [13]:
PRESET = 'base'   # small / base / large
!cd {REPO} && python -m shogi_ai.train_az \
    --data data/initial_selfplay.jsonl \
    --epochs 20 --batch-size 256 --lr 2e-3 \
    --preset {PRESET} --workers 2 \
    --out data/policy_value.pt


training on cuda
loading data
  data/initial_selfplay.jsonl: 4901 positions so far
  4901 legacy records will use a one-hot policy target
4901 positions, 85 planes, 2187 policy outputs
new network: 10 blocks x 192ch, 7.47M params
mixed precision enabled
epoch 1/20: policy=6.1051 value=0.5057 lr=2.67e-04 2.0s | val policy=6.7215 value=0.5718 top1=0.033
epoch 2/20: policy=4.3668 value=0.4659 lr=7.56e-04 0.7s | val policy=4.8998 value=0.7017 top1=0.102
epoch 3/20: policy=4.0464 value=0.4918 lr=1.35e-03 0.7s | val policy=6.3072 value=0.6092 top1=0.135
epoch 4/20: policy=3.3009 value=0.4848 lr=1.83e-03 0.7s | val policy=5.0493 value=0.5576 top1=0.176
epoch 5/20: policy=2.6564 value=0.4307 lr=2.00e-03 0.7s | val policy=3.1747 value=0.4247 top1=0.355
epoch 6/20: policy=2.2780 value=0.3738 lr=1.98e-03 0.7s | val policy=2.7161 value=0.4142 top1=0.363
epoch 7/20: policy=1.9438 value=0.3226 lr=1.91e-03 0.7s | val policy=2.5858 value=0.3299 top1=0.424
epoch 8/20: policy=1.6915 value=0.2874 lr=1.80

## 7. 学習ループを回す

1ラウンドの流れは **自己対局 → 学習 → ゲーティング対戦** です。
新しいモデルは、現行モデルに対して `--gate` 以上のスコアを取ったときだけ昇格します。
これが悪い世代でモデルを壊さないための歯止めです。

このブランチの自己対局は探索木を手ごとに再利用し、手数打ち切りの対局は
価値ターゲットに探索値を使うようになっています（どちらも自動で有効）。
探索は第3弾の **Gumbel モード**（`--search-mode gumbel`、低 sims 向け）と
**playout cap**（`--playout-cap-fraction 0.25`：全体の25%の手をフル探索して
記録し、残りは 1/4 の予算で指して記録しない）を使います。

> 並列方式の代替: `--processes` を 1 にして `--parallel-games 8` にすると、
> 第4弾の「複数対局をまたぐ推論バッチング」が効く単一プロセス方式になります。
> まずは検証済みのマルチプロセス方式で一周させ、後で切り替えて比べてください。

まずは小さく一周させる設定にしています。本格的に回すときは
`ROUNDS = 10`、`GAMES = 60` 程度に上げてください。

各ラウンドの成果物は `data/learn/` に残るので、Colab が切れても
同じコマンドを再実行すれば続きから再開します。


In [ ]:
ROUNDS = 6
GAMES = 400           # 1ラウンドの自己対局数。上のセルの実測から決める
PROCESSES = os.cpu_count()

!cd {REPO} && python -m shogi_ai.learn_loop \
    --rounds {ROUNDS} \
    --games {GAMES} \
    --simulations 200 \
    --batch-size 32 \
    --processes {PROCESSES} \
    --backend cshogi \
    --search-mode gumbel \
    --playout-cap-fraction 0.25 \
    --epochs 4 --train-batch-size 256 --lr 2e-3 \
    --max-positions 400000 \
    --preset {PRESET} \
    --eval-games 20 --gate 0.55 \
    --data-dir data/learn \
    --out-model data/policy_value.pt


In [ ]:
# 各ラウンドの昇格状況を振り返る
import json
history = json.load(open('data/learn/history.json'))
for row in history:
    score = 'n/a' if row['score'] is None else f"{row['score']:.3f}"
    flag = 'promoted' if row['promoted'] else 'rejected'
    print(f"round {row['round']:3d}  score {score:>5s}  {flag:9s}  {row['seconds']:7.1f}s")

## 8. 強さを測る

損失が下がったことは強くなった証拠になりません。実際に対戦させて確かめます。

従来型αβエンジンとの対戦は純Pythonバックエンドで行うため低速です。
局数は少なめにしてください。

In [ ]:
# 学習モデル vs ランダム初期化ネット（下限の確認）
!cd {REPO} && python -m shogi_ai.evaluate \
    --challenger data/policy_value.pt --champion random \
    --games 20 --simulations 200 --batch-size 32 \
    --backend cshogi --search-mode gumbel --preset {PRESET} 2>&1 | tail -3

In [ ]:
# 学習モデル vs 従来型αβエンジン（--backend python のため低速）
!cd {REPO} && python -m shogi_ai.evaluate \
    --challenger data/policy_value.pt --champion heuristic \
    --games 10 --simulations 200 --batch-size 16 \
    --heuristic-depth 3 --heuristic-time 1.0 \
    --backend python --search-mode gumbel --preset {PRESET} 2>&1 | tail -3

## 9. 成果物を Drive に保存する

チェックポイントと学習履歴を Drive にコピーします。ここまで済めば、
ランタイムが消えても成果物は残ります。ローカルに直接落としたい場合は
その次のセルも使えます（ローカルの `data/` に置けば、
`python3 server.py` の「学習モデル」から選べます）。


In [ ]:
import shutil, os
for name in ('policy_value.pt',):
    src = os.path.join('data', name)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(ARTIFACTS, name))
        print('saved:', os.path.join(ARTIFACTS, name))
learn_src = 'data/learn'
if os.path.isdir(learn_src):
    dst = os.path.join(ARTIFACTS, 'learn')
    if os.path.isdir(dst):
        shutil.rmtree(dst)
    shutil.copytree(learn_src, dst)
    print('saved:', dst)


In [ ]:
from google.colab import files
files.download('data/policy_value.pt')


## 調整の指針

| 症状 | 対応 |
| --- | --- |
| 自己対局が遅い | `--processes` を vCPU 数まで上げる。`--simulations` を下げる。`--resign-threshold -0.9` で決着した局を早く切る |
| GPU 使用率が低い | 正常です。CPU律速なので `--batch-size`（MCTSの葉バッチ）を32〜64に上げると多少改善します |
| ゲーティングが毎回落ちる | 1ラウンドの `--games` を増やす。`--epochs` を下げて過学習を抑える。`--lr` を半分にする |
| 引き分けばかり | `--max-moves` を伸ばす。序盤の多様性を増やすため `--temperature-moves` を上げる |
| VRAM が余っている | `--preset large`（15ブロック×256ch）に上げる。ただし推論が遅くなるぶん自己対局も遅くなります |
| 途中で切れた | 同じ `learn_loop` コマンドを再実行すれば、既存のラウンド成果物を再利用して続きから走ります |

**規模の目安**: policy の top-1 一致率が 0.3 を超えるあたりから、
従来型αβエンジン（深さ3）に対して勝てるようになってきます。
そこに到達するには数万局規模の自己対局が必要です。